In [3]:
import json
import pandas as pd
import numpy as np
import torch
import os
import csv
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from underthesea import ner, word_tokenize
import warnings

c:\Users\quang\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\quang\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [13]:
# Tắt cảnh báo khi tính PPL
warnings.filterwarnings("ignore", category=FutureWarning)

# =================================================================
#               KHỞI TẠO MÔ HÌNH VÀ TOKENIZER
# =================================================================

# 1. Mô hình nhúng (EMBEDDING) - Vẫn dùng PhoBERT (AutoModel)
EMB_MODEL_NAME = "vinai/phobert-base"
tokenizer_emb = AutoTokenizer.from_pretrained(EMB_MODEL_NAME)
model_emb = AutoModel.from_pretrained(EMB_MODEL_NAME)

# 2. Mô hình cho Perplexity (PPL) - SỬ DỤNG CAUSAL LANGUAGE MODEL (CLM)
# LƯU Ý: Đây là mô hình tiếng Anh. Bạn nên thay thế bằng CLM tiếng Việt nếu có.
PPL_MODEL_NAME = "gpt2" 
tokenizer_ppl = AutoTokenizer.from_pretrained(PPL_MODEL_NAME)
model_ppl = AutoModelForCausalLM.from_pretrained(PPL_MODEL_NAME) # Dùng AutoModelForCausalLM

# Đặt thiết bị tính toán (GPU nếu có)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_emb.to(DEVICE)
model_ppl.to(DEVICE)

# Thêm padding token cho CLM nếu thiếu (cần thiết cho mô hình GPT)
if tokenizer_ppl.pad_token is None:
    tokenizer_ppl.pad_token = tokenizer_ppl.eos_token

In [14]:
def get_embedding(tokenizer, model, text, qid=None, subject=None, max_len=512):
    """
    Tính toán Semantic Embedding (CLS Token). 
    Đảm bảo đầu ra luôn là mảng 1 chiều (768,) hoặc (0,) nếu rỗng.
    """
    if not text.strip():
        return np.zeros(768, dtype=np.float32) # Luôn trả về 768 chiều
    try:
        clean_text = text.replace("_", " ")
        inputs = tokenizer(clean_text, return_tensors="pt", truncation=True, padding=True, max_length=max_len).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Lấy CLS token và đảm bảo nó được làm phẳng thành 1D (768,)
        # Sử dụng .flatten() để đảm bảo chỉ có 768 phần tử
        return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy().flatten()
        
    except Exception as e:
        # print(f"Lỗi khi embedding câu hỏi id={qid}, môn={subject}: {e}")
        return np.zeros(768, dtype=np.float32)

In [15]:
def calculate_jaccard_sim(text1, text2):
    """Tính Jaccard Similarity (Tương đồng Từ vựng)."""
    tokens1 = set(text1.replace("_", " ").split())
    tokens2 = set(text2.replace("_", " ").split())
    if not tokens1 and not tokens2:
        return 1.0
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1.intersection(tokens2)) / len(tokens1.union(tokens2))

In [16]:
def calculate_entity_overlap(text1, text2):
    """Tính tỷ lệ trùng lặp thực thể (Entity Overlap)."""
    text1_clean = text1.replace("_", " ")
    text2_clean = text2.replace("_", " ")
    
    # Chỉ xem xét các thực thể chính
    entities1 = set([item[0] for item in ner(text1_clean) if item[1] in ['PERSON', 'LOCATION', 'ORGANIZATION', 'EVENT']])
    entities2 = set([item[0] for item in ner(text2_clean) if item[1] in ['PERSON', 'LOCATION', 'ORGANIZATION', 'EVENT']])
    
    if not entities1 and not entities2:
        return 0.0
    if not entities1 or not entities2:
        return 0.0
    
    return len(entities1.intersection(entities2)) / len(entities1.union(entities2))

In [17]:
def calculate_ppl(tokenizer, model, text, max_len=512):
    """
    Tính Perplexity (PPL) bằng CLM. Hàm này KHÔNG cần thay đổi, 
    vì mô hình đã được chuyển sang CLM (AutoModelForCausalLM) ở phần khởi tạo.
    """
    if not text.strip():
        return 10000.0
    try:
        clean_text = text.replace("_", " ")
        # Sử dụng padding token của CLM
        encodings = tokenizer(clean_text, return_tensors='pt', truncation=True, max_length=max_len, padding=True).to(DEVICE)
        
        input_ids = encodings.input_ids
        labels = input_ids.clone()
        
        with torch.no_grad():
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss
            
        return torch.exp(loss).item()
    except Exception as e:
        text_len = len(text.split())
        return max(100.0, text_len * 10.0)

# Các từ khóa Cấu trúc (Structure)
WH_WORDS = {
    "is_Why": ["Tại sao", "Lý do", "Nguyên nhân"],
    "is_How": ["Như thế nào", "Cách thức", "Biện pháp"],
    "is_When": ["Khi nào", "Thời điểm", "Năm"],
    "is_Compare": ["So sánh", "Điểm khác biệt", "Giống nhau"]
}
NEGATION_WORDS = ["không", "ngoại trừ", "sai", "chưa", "ít", "ko", "chứ không phải"]

In [18]:
def extract_structure_features(question_stem):
    """Trích xuất các features cấu trúc câu hỏi (Wh-words, Phủ định)."""
    features = {}
    q_lower = question_stem.lower()
    
    for key, words in WH_WORDS.items():
        features[key] = 1 if any(word.lower() in q_lower for word in words) else 0
        
    features["is_Negation"] = 1 if any(word.lower() in q_lower for word in NEGATION_WORDS) else 0
    
    return features

In [19]:
def get_diff(input_file, output_dir):
    
    # --- KHỞI TẠO TÊN CỘT FEATURES ---
    base_cols = ["id", "subject", "question", "answer"]
    sim_cols = ["mean_semantic_sim", "max_sim", "min_sim", "std_sim", "range_sim",
                "mean_jaccard_sim", "std_jaccard_sim",
                "mean_entity_overlap", "max_entity_overlap",
                "mean_inter_semantic_sim", "std_inter_semantic_sim"]
    ppl_cols = ["PPL_Stem", "PPL_Option_Mean", "PPL_Gap", "PPL_Range"]
    structure_cols = list(WH_WORDS.keys()) + ["is_Negation"]
    fieldnames = base_cols + sim_cols + ppl_cols + structure_cols
    # --------------------------------

    all_features = []; 
    with open(input_file, "r", encoding="utf-8") as f: data = json.load(f)
    print(f"---------Đang tính toán độ nhiễu nâng cao cho {len(data)} câu hỏi-------------\n")
    
    for item in tqdm(data):
        question = item["question"]; options = item["options"]; answer_key = item["answer"]; qid = item["id"]; subject = item.get("subject", "Unknown")
        current_row = {"id": qid, "subject": subject, "question": question, "answer": answer_key}

        # --- I. Cấu trúc & PPL (Sử dụng CLM) ---
        current_row.update(extract_structure_features(question))
        ppl_stem = calculate_ppl(tokenizer_ppl, model_ppl, question)
        ppl_options = [calculate_ppl(tokenizer_ppl, model_ppl, opt) for opt in options]
        
        current_row["PPL_Stem"] = ppl_stem
        if ppl_options:
            ppl_option_mean = np.mean(ppl_options); current_row["PPL_Option_Mean"] = ppl_option_mean
            current_row["PPL_Range"] = np.max(ppl_options) - np.min(ppl_options); current_row["PPL_Gap"] = abs(ppl_stem - ppl_option_mean)
        else:
            current_row["PPL_Option_Mean"] = current_row["PPL_Range"] = current_row["PPL_Gap"] = 0.0
            
        # --- II. Tương đồng Đa tầng ---
        try:
            answer_index = options.index(answer_key); correct_option_text = options[answer_index]
        except ValueError: continue 
        
        jaccard_sims, entity_overlaps, semantic_sims = [], [], []; wrong_embs = []
        emb_answer = get_embedding(tokenizer_emb, model_emb, question + " " + correct_option_text, qid=qid, subject=subject)
        if not emb_answer.any(): continue

        for i, opt in enumerate(options):
            if i != answer_index:
                emb_distractor = get_embedding(tokenizer_emb, model_emb, question + " " + opt, qid=qid, subject=subject)
                wrong_embs.append(emb_distractor)
                sim = cosine_similarity(emb_answer.reshape(1, -1), emb_distractor.reshape(1, -1))[0][0]; semantic_sims.append(sim)
                jaccard_sims.append(calculate_jaccard_sim(correct_option_text, opt))
                entity_overlaps.append(calculate_entity_overlap(correct_option_text, opt))

        # 2. Tổng hợp Sim giữa Đúng và Sai
        if semantic_sims:
            current_row["mean_semantic_sim"] = float(np.mean(semantic_sims)); current_row["max_sim"] = float(np.max(semantic_sims)); current_row["min_sim"] = float(np.min(semantic_sims)); current_row["std_sim"] = float(np.std(semantic_sims)); current_row["range_sim"] = current_row["max_sim"] - current_row["min_sim"]
            current_row["mean_jaccard_sim"] = float(np.mean(jaccard_sims)); current_row["std_jaccard_sim"] = float(np.std(jaccard_sims))
            current_row["mean_entity_overlap"] = float(np.mean(entity_overlaps)); current_row["max_entity_overlap"] = float(np.max(entity_overlaps))
            
        # 3. Tương đồng giữa các Nhiễu
        inter_sims = []; num_wrong = len(wrong_embs)
        for i in range(num_wrong):
            for j in range(i + 1, num_wrong):
                sim = cosine_similarity(wrong_embs[i].reshape(1, -1), wrong_embs[j].reshape(1, -1))[0][0]; inter_sims.append(sim)
        
        if inter_sims:
            current_row["mean_inter_semantic_sim"] = float(np.mean(inter_sims)); current_row["std_inter_semantic_sim"] = float(np.std(inter_sims))
        else:
            current_row["mean_inter_semantic_sim"] = 0.0; current_row["std_inter_semantic_sim"] = 0.0
            
        all_features.append(current_row)

    # --- III. Lưu kết quả ---
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, "noise_features_advanced.csv")
    
    processed_rows = []
    for row in all_features:
        full_row = {col: row.get(col, 0.0) for col in fieldnames}
        full_row["id"] = row.get("id"); full_row["subject"] = row.get("subject"); full_row["question"] = row.get("question"); full_row["answer"] = row.get("answer")
        processed_rows.append(full_row)
        
    with open(csv_path, "w", newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(processed_rows)

    print(f"--------Đã hoàn thành xong feartures tính độ nhiễu nâng cao-------------\n", csv_path)
    print(f"Tổng số câu hỏi được xử lý: {len(processed_rows)}")
    
    return csv_path

In [20]:
input_file = r"Output_ws\questions.json" 
output_dir = r"Output_ws\features" # Đặt thư mục đầu ra

# Tải dữ liệu từ file JSON
try:
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Đã tải thành công {len(data)} câu hỏi từ {input_file}")
except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file tại đường dẫn: {input_file}")
    exit()

final_csv_path = get_diff(input_file, output_dir)
print(f"\n✅ Hoàn thành xử lý. Dữ liệu Feature đã được lưu tại: {final_csv_path}")

Đã tải thành công 3097 câu hỏi từ Output_ws\questions.json
---------Đang tính toán độ nhiễu nâng cao cho 3097 câu hỏi-------------



100%|██████████| 3097/3097 [40:47<00:00,  1.27it/s]   

--------Đã hoàn thành xong feartures tính độ nhiễu nâng cao-------------
 Output_ws\features\noise_features_advanced.csv
Tổng số câu hỏi được xử lý: 3059

✅ Hoàn thành xử lý. Dữ liệu Feature đã được lưu tại: Output_ws\features\noise_features_advanced.csv


In [10]:

print(f"--------------------------------------Đang kết hợp data--------------------------------------")
df1 = pd.read_csv(r"Output_ws\features\noise_features_advanced.csv")
df3 = pd.read_csv(r"..\training\data_for_training\van_with_bloom_out.csv")
merged = pd.merge(df1, df3, on = "id", how = "inner")
result = merged.drop(columns=["question_x", "question_y", "subject", "answer"])
result.to_csv("merged.csv", index = False)
print("Done.........")

--------------------------------------Đang kết hợp data--------------------------------------
Done.........


In [ ]:
import numpy as np
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ==================================================================
# 1. KHỞI TẠO MÔ HÌNH CLM
# ==================================================================

# Sử dụng mô hình CLM phổ biến (GPT-2) để tính PPL.
# LƯU Ý: Bạn nên thay thế bằng mô hình Causal LM tiếng Việt (như phobert-base-v2 nếu được fine-tune) 
# để có kết quả chính xác cho data tiếng Việt.
MODEL_NAME_CLM = "gpt2" # Ví dụ: Thay thế bằng mô hình CLM tiếng Việt của bạn
tokenizer_clm = AutoTokenizer.from_pretrained(MODEL_NAME_CLM)
model_clm = AutoModelForCausalLM.from_pretrained(MODEL_NAME_CLM)

# Đặt thiết bị tính toán
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_clm.to(DEVICE)

# Thêm padding token nếu tokenizer không có (cần thiết cho mô hình GPT)
if tokenizer_clm.pad_token is None:
    tokenizer_clm.pad_token = tokenizer_clm.eos_token
    
# ==================================================================
# 2. HÀM TÍNH PPL CHUẨN (CLM)
# ==================================================================

def calculate_ppl_clm(tokenizer, model, text, max_len=512):
    """
    Tính Perplexity (PPL) bằng mô hình Causal Language Model (CLM).
    """
    if not text.strip():
        return 10000.0 # PPL rất cao cho văn bản rỗng
    
    try:
        # Xử lý text (bỏ dấu gạch dưới nếu cần)
        clean_text = text.replace("_", " ")
        
        # Mã hóa đầu vào
        encodings = tokenizer(clean_text, return_tensors='pt', 
                              truncation=True, padding=True, 
                              max_length=max_len).to(DEVICE)
        
        input_ids = encodings.input_ids
        labels = input_ids.clone()
        
        # Tính toán loss (Negative Log Likelihood)
        with torch.no_grad():
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss
            
        # PPL = exp(loss)
        return torch.exp(loss).item()
        
    except Exception as e:
        print(f"Lỗi khi tính PPL cho đoạn văn bản: {e}")
        return 5000.0

# ==================================================================
# 3. HÀM CHÍNH: TÍNH TẤT CẢ PPL CHO DATA CỦA BẠN
# ==================================================================

def get_ppl_features(data):
    """
    Tính PPL cho Question Stem và Options, và các Features liên quan.
    """
    ppl_results = []

    for item in tqdm(data, desc="Tính toán PPL"):
        question = item["question"]
        options = item["options"]
        qid = item["id"]

        result = {"id": qid}
        
        # 1. PPL của Stem
        ppl_stem = calculate_ppl_clm(tokenizer_clm, model_clm, question)
        result["PPL_Stem"] = ppl_stem

        # 2. PPL của Options
        ppl_options = []
        for opt in options:
            # Bỏ chữ cái đầu (A., B.,...) trước khi tính PPL
            option_text = opt.split('. ', 1)[-1]
            ppl_opt = calculate_ppl_clm(tokenizer_clm, model_clm, option_text)
            ppl_options.append(ppl_opt)
        
        if ppl_options:
            ppl_option_mean = np.mean(ppl_options)
            ppl_option_max = np.max(ppl_options)
            ppl_option_min = np.min(ppl_options)
            
            result["PPL_Option_Mean"] = float(ppl_option_mean)
            result["PPL_Range"] = float(ppl_option_max - ppl_option_min)
            result["PPL_Gap"] = float(abs(ppl_stem - ppl_option_mean))
            result["PPL_Max_Option"] = float(ppl_option_max) # Thêm PPL Max Option
        else:
            result["PPL_Option_Mean"] = result["PPL_Range"] = result["PPL_Gap"] = result["PPL_Max_Option"] = 0.0

        ppl_results.append(result)
        
    return ppl_results

# ==================================================================
# 4. DỮ LIỆU MẪU (SAMPLE DATA) VÀ THỰC THI
# ==================================================================

sample_data = [
    {
        "id": 1001,
        "subject": "Văn_học",
        "question": "Trong đoạn trích trên, tác_giả sử_dụng hình_thức nghệ_thuật chính nào?",
        "options": ["A. Phép đối xứng", "B. Liệt_kê", "C. Ẩn_dụ và hoán_dụ"],
        "answer": "C. Ẩn_dụ và hoán_dụ"
    },
    {
        "id": 1002,
        "subject": "Lịch_sử",
        "question": "Nguyên_nhân sâu_xa dẫn đến sự_kiện Chiến_tranh thế_giới thứ nhất là gì? Đây là một câu hỏi có từ ngữ rất hàn lâm và phức tạp về mặt thuật ngữ chuyên_ngành.",
        "options": ["A. Sự_kiện ám_sát tại Sarajevo", "B. Mâu_thuẫn gay_gắt giữa các nước đế_quốc về thị_trường và thuộc_địa", "C. Các hiệp_ước phòng_thủ chung được ký_kết"],
        "answer": "B. Mâu_thuẫn gay_gắt giữa các nước đế_quốc về thị_trường và thuộc_địa"
    },
]

ppl_features = get_ppl_features(sample_data)

print("\n--- KẾT QUẢ TÍNH TOÁN PPL (CLM) ---")
for result in ppl_features:
    print(f"\nID: {result['id']}")
    print(f"  PPL_Stem (Câu hỏi): {result['PPL_Stem']:.2f}")
    print(f"  PPL_Option_Mean (Trung bình đáp án): {result['PPL_Option_Mean']:.2f}")
    print(f"  PPL_Gap (Chênh lệch): {result['PPL_Gap']:.2f}")
    print(f"  PPL_Max_Option (Đáp án khó nhất): {result['PPL_Max_Option']:.2f}")

c:\Users\quang\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\quang\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Tính toán PPL: 100%|██████████| 2/2 [00:00<00:00,  2.19it/s]


--- KẾT QUẢ TÍNH TOÁN PPL (CLM) ---

ID: 1001
  PPL_Stem (Câu hỏi): 36.86
  PPL_Option_Mean (Trung bình đáp án): 76.40
  PPL_Gap (Chênh lệch): 39.54
  PPL_Max_Option (Đáp án khó nhất): 101.94

ID: 1002
  PPL_Stem (Câu hỏi): 26.16
  PPL_Option_Mean (Trung bình đáp án): 33.66
  PPL_Gap (Chênh lệch): 7.50
  PPL_Max_Option (Đáp án khó nhất): 41.30
